In [1]:
# 노트북이 실행 중인 파이썬 환경에 BigQuery 관련 패키지를 설치함
%pip install google-cloud-bigquery pandas-gbq db-dtypes pyarrow pydata-google-auth

Note: you may need to restart the kernel to use updated packages.


In [2]:
from google.cloud import bigquery
import pydata_google_auth

PROJECT_ID = "dailyglow-analytics-514401"   # 2단계에서 메모한 프로젝트 ID (이름 아님)
LOCATION = "asia-northeast3"         # 데이터셋을 만든 리전 (서울)

# 브라우저 로그인 창을 띄워 내 Google 계정으로 인증함
# 처음 한 번만 로그인하면 이후에는 컴퓨터에 저장된 인증 정보를 재사용함 (리포지토리 밖에 저장됨)
credentials = pydata_google_auth.get_user_credentials(
    ["https://www.googleapis.com/auth/bigquery"]
)

# 인증 정보로 BigQuery 클라이언트 생성
# location을 지정해야 쿼리가 서울 리전에서 실행됨 (안 하면 기본값 US에서 데이터셋을 찾다가 에러)
client = bigquery.Client(project=PROJECT_ID, credentials=credentials, location=LOCATION)

print(client.project)  # 프로젝트 ID가 그대로 출력되면 연결 성공

dailyglow-analytics-514401


In [3]:
# 프로젝트에 있는 데이터셋 목록을 하나씩 꺼내 이름(dataset_id)을 출력함
for dataset in client.list_datasets():
    print(dataset.dataset_id)

mart
raw
stg


In [4]:
import pandas as pd

df = pd.DataFrame({
    'user_id': [101, 102, 103],
    'age': [25, 30, 28],
    'purchase': [True, False, True]
})


client.load_table_from_dataframe(df, f"{PROJECT_ID}.raw.test_table").result()



LoadJob<project=dailyglow-analytics-514401, location=asia-northeast3, id=6f5c1de4-6c8b-4d39-a719-6c7bc37ec707>

In [5]:
# create_bqstorage_client=False: Storage API를 쓰지 않고 기본(REST) 방식으로 결과를 받겠다고 명시 → 경고가 사라짐
client.query("select * from raw.test_table").to_dataframe(create_bqstorage_client=False)

,user_id,age,purchase
0,102,30,False
1,101,25,True
2,103,28,True


In [6]:
# 프로젝트ID.데이터셋.테이블 경로로 테스트 테이블을 삭제함
# not_found_ok=True: 테이블이 없어도 에러를 내지 않음 (셀을 여러 번 실행해도 안전)
client.delete_table(f"{PROJECT_ID}.raw.test_table", not_found_ok=True)